# Download MODIS Heat Detection Data from WIFIRE Firemap
### What this notebook does:

1. Search for MODIS heat detections by datetime range
2. Download all detection features from the WIFIRE Firemap Geoserver
3. Visualize heat detections on a Folium map
4. Save as JSON, CSV, and/or zipped Shapefile


### Data source:

* WIFIRE Firemap - MODIS heat detection data
* Layer: view_modis_c6
* Contains actual mapped detections with timestamps

### Requirements:

* Internet connection
* Fire name
* Fire start/stop datetimes

# Dependencies

In [1]:
import requests
import json
import sys
from datetime import datetime
import folium
import os

# Fire Data Collection


Testing the Geoserver API Query

In [2]:
# ========== USER INPUTS ==========

fire_name = "Border 2"  # Your chosen fire name will be used for download's filename

# Bounding Box (syntax: "LowerLeft_Longitude,LowerLeft_Latitude,UpperRight_Longitude,UpperRight_Latitude,'EPSG:Code'")
LL_lon = -116.93191
LL_lat = 32.56482
UR_lon = -116.81826
UR_lat = 32.67451
bbox = f"{LL_lon},{LL_lat},{UR_lon},{UR_lat},'EPSG:4326'"  # For a complete CA BBOX, enter: "-124.5,32.45,-114.0,42.10,'EPSG:4326'"
map_center = [(LL_lat+UR_lat)/2.0, (LL_lon+UR_lon)/2.0]

# Specify a datetime & epoch (yyyy, m, d, h, m, s)
dt_start = datetime(2010, 1, 1, 0, 0, 1)    # Note: the first available MODIS detection starts from (2017, 5, 20, 21, 35, 0)
dt_end = datetime(2025, 2, 1, 23, 59, 59)   # Note: Large datetime ranges (ex: more than 1 year) may overtax your computer's resources

# ==================================

In [3]:
# Convert datetime to epoch
epoch_start = int(dt_start.timestamp())
epoch_end = int(dt_end.timestamp())

# GeoServer WFS URL configuration
url = "https://firemap.sdsc.edu/geoserver/wfs"
params = {
    'service': 'WFS',
    'version': '1.0.0',
    'request': 'GetFeature',
    'maxFeatures': '100000',
    'exceptions': 'application/json',
    'typeName': 'WIFIRE:view_modis_c6',
    'CQL_FILTER': f'epoch_time BETWEEN {epoch_start} AND {epoch_end} AND BBOX(location,{bbox})',
    'outputFormat': 'application/json'
}

# Request the data
r = requests.get(url, params=params, timeout=60)
if r.status_code != 200:
    print(r.text)
    sys.exit(1)

j = r.json()
f = j.get("features", [])

if not f:
    raise ValueError(
        f"No MODIS dections found for '{fire_name}'.\n"
        f"Check that the datetime range is valid.\n"
        f"Hint: the first available MODIS detection starts from 2016-08-11T09:20:00Z"
    )

print(f"✓ Found {len(f)} detections \n")

# View the JSON
print(json.dumps(j,indent=2))   # WARNING: Large datetime ranges may overtax your computer's resources

✓ Found 42 detections 

{
  "type": "FeatureCollection",
  "features": [
    {
      "type": "Feature",
      "id": "view_modis_c6.fid-20d8c1eb_19db0ce87e1_5aac",
      "geometry": {
        "type": "Point",
        "coordinates": [
          -116.82,
          32.643
        ]
      },
      "geometry_name": "location",
      "properties": {
        "seconds_ago": 281472616.5245991,
        "epoch_time": 1495316100,
        "brightness": 331.1,
        "scan": 2.6,
        "track": 1.5,
        "acq_date": "2017-05-20Z",
        "acq_time": "1970-01-01T21:35:00Z",
        "satellite": "A",
        "confidence": 44,
        "version": "6.0NRT",
        "bright_t31": 314.7,
        "frp": 59.2,
        "daynight": "D"
      }
    },
    {
      "type": "Feature",
      "id": "view_modis_c6.fid-20d8c1eb_19db0ce87e1_5aad",
      "geometry": {
        "type": "Point",
        "coordinates": [
          -116.821,
          32.657
        ]
      },
      "geometry_name": "location",
      "

In [4]:
# # For full Traceback 
# %tb

Previewing the JSON response in GIS

In [5]:
# Create map object
m = folium.Map(location=map_center, tiles='cartodbpositron', zoom_start=11)

# Add 375m Radius Circle Marker for GeoJson features
features = folium.features.GeoJson(j,
    marker=folium.Circle(
        radius=500, # Radius in meters
        fill=True,
        fill_color="blue",
        fill_opacity=0.2),
        tooltip = folium.GeoJsonTooltip(
            fields=['acq_date', 'confidence', 'daynight'])
).add_to(m)

# Preview map
m   # WARNING: Large datetime ranges may overtax your computer's resources


Create Output Filename & Directory

In [ ]:
# Set Output folder & filename prefix
filename = fire_name.replace(" ", "_")
os.makedirs(f"./Fires/{filename}", exist_ok=True)

print(filename)

Getting MODIS JSON

In [ ]:
# GeoServer WFS URL JSON configuration
url = "https://firemap.sdsc.edu/geoserver/wfs"
params = {
    'service': 'WFS',
    'version': '1.0.0',
    'request': 'GetFeature',
    'maxFeatures': '100000',
    'exceptions': 'application/json',
    'typeName': 'WIFIRE:view_modis_c6',
    'CQL_FILTER': f'epoch_time BETWEEN {epoch_start} AND {epoch_end} AND BBOX(location,{bbox})',
    'outputFormat': 'application/json'
}

r = requests.get(url, params=params, timeout=60)
if r.status_code != 200:
    print(r.text)
    sys.exit(1)
elif r.status_code == 200:
    j = r.json()    
    with open(f"./Fires/{filename}/{filename}_modis.json", "w", encoding="utf-8") as file:
        json.dump(j, file)
    print("JSON saved successfully.")
else:
    print("Error:", r.status_code, r.text)

Getting MODIS CSV

In [ ]:
# # GeoServer WFS URL CSV configuration
# url = "https://firemap.sdsc.edu/geoserver/wfs"
# params = {
#     'service': 'WFS',
#     'version': '1.0.0',
#     'request': 'GetFeature',
#     'maxFeatures': '100000',
#     'exceptions': 'application/json',
#     'typeName': 'WIFIRE:view_modis_c6',
#     'CQL_FILTER': f'epoch_time BETWEEN {epoch_start} AND {epoch_end} AND BBOX(location,{bbox})',
#     'outputFormat': 'csv'
# }

# r = requests.get(url, params=params, timeout=60)
# if r.status_code != 200:
#     print(r.text)
#     sys.exit(1)
# elif r.status_code == 200:
#     with open(f"./Fires/{filename}/{filename}_modis.csv", "wb") as f:
#         f.write(r.content)
#     print("CSV saved successfully.")
# else:
#     print("Error:", r.status_code, r.text)

Getting MODIS Shapefiles

In [ ]:
# # GeoServer WFS URL CSV configuration
# url = "https://firemap.sdsc.edu/geoserver/wfs"
# params = {
#     'service': 'WFS',
#     'version': '1.0.0',
#     'request': 'GetFeature',
#     'maxFeatures': '100000',
#     'exceptions': 'application/json',
#     'typeName': 'WIFIRE:view_modis_c6',
#     'CQL_FILTER': f'epoch_time BETWEEN {epoch_start} AND {epoch_end} AND BBOX(location,{bbox})',
#     'outputFormat': 'shape-zip'
# }

# r = requests.get(url, params=params, timeout=60)
# if r.status_code != 200:
#     print(r.text)
#     sys.exit(1)
# elif r.status_code == 200:
#     with open(f"./Fires/{filename}/{filename}_modis.zip", "wb") as f:
#         for chunk in r.iter_content(chunk_size=1024*1024):
#             f.write(chunk)
#     print("Shapefile ZIP saved successfully.")
# else:
#     print("Error:", r.status_code, r.text)


